# 06 · Early-stopping sweep

**Purpose.** Register the early-stopping checkpoints (C2: 1–200 epochs, C6: 5–100 epochs) as sweep points and trace the accuracy–privacy frontier.

**Inputs.** Checkpointed targets and their attack runs (notebooks 02–05).

**Outputs.** `sweep_points` rows (x = epoch), figure F3.

**Syllabus topics.** Discriminant-function classification; lightweight models; generalisation vs memorisation.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
import matplotlib.pyplot as plt

from mia import db, plots
from mia.config import DB_PATH, load_experiment
from mia.training import register_sweep_points

exp = load_experiment()
conn = db.connect(DB_PATH)
register_sweep_points(conn, exp)
es = db.sweep_df(conn, "early_stopping")
es.groupby(["base_config_id", "x_value"])[["train_acc", "test_acc", "gap_acc", "auc", "tpr_at_1pct_fpr", "tpr_at_01pct_fpr"]].mean().round(4)

In [ ]:
plots.apply_style()
fig = plots.f3_early_stopping(es)
print("saved:", *plots.save_figure(fig, "F3_early_stopping"))
plt.show()

In [ ]:
for base, g in es.groupby("base_config_id"):
    a = g.groupby("x_value")[["test_acc", "auc"]].mean()
    print(f"{base}: AUC {a['auc'].iloc[0]:.3f} at epoch {a.index[0]:.0f} -> {a['auc'].iloc[-1]:.3f} at epoch {a.index[-1]:.0f}; "
          f"test acc {a['test_acc'].iloc[0]:.3f} -> {a['test_acc'].iloc[-1]:.3f}")

## Results
The printout shows how attack AUC and test accuracy move from the first to the last checkpoint of each base; F3 shows the full frontier.